# EN3150 A03 - canonical Colab runner

This notebook is deliberately a thin driver. It caches EuroSAT in Google Drive and calls the same `scripts/` entrypoints used locally; model, data, training, evaluation, and profiling logic stays in `src/pr_edge/`.

Before running all cells:

1. Select **Runtime > Change runtime type > T4 GPU**.
2. In Colab's **Secrets** panel, add `GITHUB_TOKEN` with read access to this private repository and enable notebook access. Never paste the token into a cell.
3. Review the configuration cell below. For graded comparison runs, keep every model on the same Colab GPU type.


In [ ]:
# @title Run configuration
REPO_URL = "https://github.com/dpiyumal2319/pattern-recognition-cnn-edge.git"
BRANCH = "main"
CONFIG_PATH = "configs/model_b.yaml"
RUN_PREPARE_DATA = True
SMOKE = False
DRIVE_FOLDER = "EN3150_A03"


## 1. Mount Drive and check out the repository

The token is read from Colab Secrets and sent to GitHub only as a transient HTTP header. It is not written into the clone URL or Git configuration.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER
REPO_DIR = Path("/content/pattern-recognition-cnn-edge")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
from base64 import b64encode
import subprocess
from google.colab import userdata

github_token = userdata.get("GITHUB_TOKEN").strip()
if not github_token:
    raise RuntimeError("Colab secret GITHUB_TOKEN is empty")
credentials = b64encode(f"x-access-token:{github_token}".encode()).decode()
git = ["git", "-c", f"http.extraHeader=AUTHORIZATION: basic {credentials}"]

if (REPO_DIR / ".git").is_dir():
    subprocess.run([*git, "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run([*git, "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run([*git, "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run([*git, "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)

del github_token, credentials, git


## 2. Attach the persistent EuroSAT cache

Only the downloaded `data/eurosat/` directory lives in Drive. The committed split manifest and normalization contract remain in the clone, so every run uses the repository version.


In [ ]:
CACHE_EUROSAT = DRIVE_ROOT / "data" / "eurosat"
REPO_EUROSAT = REPO_DIR / "data" / "eurosat"
CACHE_EUROSAT.mkdir(parents=True, exist_ok=True)

if REPO_EUROSAT.is_symlink():
    if REPO_EUROSAT.resolve() != CACHE_EUROSAT.resolve():
        raise RuntimeError(f"Unexpected dataset link: {REPO_EUROSAT}")
elif REPO_EUROSAT.exists():
    raise RuntimeError(f"Refusing to replace existing local dataset: {REPO_EUROSAT}")
else:
    REPO_EUROSAT.symlink_to(CACHE_EUROSAT, target_is_directory=True)

print(f"EuroSAT cache: {CACHE_EUROSAT}")


## 3. Install and verify the runtime


In [ ]:
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO_DIR, check=True)


In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA is unavailable. Do not compare epoch times from this runtime with GPU runs.")


## 4. Prepare data and run the selected experiment

The first data-preparation run downloads about 94 MB. Later runtimes reuse the Drive cache. Set `SMOKE = True` to check the wiring without reading EuroSAT.


In [ ]:
if RUN_PREPARE_DATA and not SMOKE:
    # The split manifest is committed, so prepare_data intentionally reuses it.
    # Bootstrap the Drive-backed image cache separately on a first Colab run.
    from torchvision.datasets import EuroSAT

    image_root = REPO_DIR / "data" / "eurosat" / "2750"
    if not image_root.exists():
        print("Downloading EuroSAT into the persistent Drive cache...")
        EuroSAT(root=str(REPO_DIR / "data"), download=True)

    subprocess.run(
        [sys.executable, "scripts/prepare_data.py", "--config", "configs/base.yaml"],
        cwd=REPO_DIR,
        check=True,
    )
else:
    print("Skipping real-data preparation.")


In [ ]:
experiment = [sys.executable, "scripts/run_experiment.py", "--config", CONFIG_PATH]
if SMOKE:
    experiment.append("--smoke")
subprocess.run(experiment, cwd=REPO_DIR, check=True)


## 5. Persist the run artifacts

The repository writes the results contract under `/content`. This final cell mirrors that run directory to Drive so checkpoints and metrics survive a runtime disconnect.


In [ ]:
import shutil

sys.path.insert(0, str(REPO_DIR / "src"))
from pr_edge.config import load_config

resolved = load_config(REPO_DIR / CONFIG_PATH)
run_name = f"_smoke_{resolved.run_id}" if SMOKE else resolved.run_id
source_run = REPO_DIR / "results" / run_name
drive_run = DRIVE_ROOT / "results" / run_name
shutil.copytree(source_run, drive_run, dirs_exist_ok=True)
print(f"Persistent artifacts: {drive_run}")
